## Работа с данными и обучение модели

### 1. Обработка данных
(Создание DataLoader and Dataset)


In [8]:
import os
import torch
import torch.optim as optim
import torch.nn as  nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd
import numpy as np
from PIL import Image



DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_SIZE = 5000
IMAGE_SIZE = 100
images_folder = 'public_tests/00_test_img_input/train/images'
answers_folder = 'public_tests/00_test_img_input/train/gt.csv'


In [9]:
class FaceDataset(Dataset):
    def __init__(self, images_folder, answers_folder, image_size, device, train_size=5000, train=True):
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std= [0.229, 0.224, 0.225])
        ])
        gt = pd.read_csv(answers_folder)
        if train:
            names = gt.iloc[:train_size, 0].tolist()
            pts = gt.iloc[:train_size, 1:].to_numpy(dtype='float32')
        else:
            names = gt.iloc[train_size:, 0].tolist()
            pts = gt.iloc[train_size:, 1:].to_numpy(dtype='float32')
        images = []
        points = []

        for i, name in enumerate(names):
            path = os.path.join(images_folder, name)
            img = Image.open(path).convert('RGB')
            images.append(transform(img))

            orig_w, orig_h = img.size
            kx, ky = image_size / orig_w, image_size / orig_h

            point = pts[i].copy()
            point[0::2] *= kx
            point[1::2] *= ky
            points.append(torch.from_numpy(point))

        self.images = torch.stack(images).to(device)
        self.points = torch.stack(points).to(device)


    def __len__(self):
        return len(self.images)


    def __getitem__(self, idx):
        return self.images[idx], self.points[idx]






In [10]:
train_ds = FaceDataset(images_folder, answers_folder, IMAGE_SIZE, DEVICE, TRAIN_SIZE, train=True)
val_ds = FaceDataset(images_folder, answers_folder, IMAGE_SIZE, DEVICE, TRAIN_SIZE, train=False)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64,  shuffle=False, drop_last=True)

### 2. Базовая модель

In [11]:
class FaceNet(nn.Module):
    def __init__(self, image_size):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(128),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(256),  nn.ReLU(), nn.MaxPool2d(2)
        )

        side = image_size // 8
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=side * side * 256, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=28)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

In [12]:
model = FaceNet(IMAGE_SIZE).to(DEVICE)

### 3. Обучение базовой модели
Опишем функцию train

In [13]:
def train(model, train_loader, val_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_function = nn.MSELoss()


    for epoch in range(epochs):
        losses_train = []
        losses_val = []

        model.train()
        for images, labels in train_loader:
            preds = model(images)
            loss = loss_function(preds, labels)
            loss.backward()
            losses_train.append(loss.item())
            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            for images, labels in val_loader:
                preds = model(images)
                loss = loss_function(preds, labels)
                losses_val.append(loss.item())


        print(f'Epoch:{epoch + 1}  Train_loss:{np.mean(losses_train)}   Test_loss:{np.mean(losses_val)}')






In [14]:
train(model, train_loader, val_loader, 30)

Epoch:1  Train_loss:175.99191383215097   Test_loss:33.936363983154294
Epoch:2  Train_loss:33.96789948756878   Test_loss:30.021516609191895
Epoch:3  Train_loss:31.868939509758583   Test_loss:28.286507161458335
Epoch:4  Train_loss:26.019812155992557   Test_loss:23.92574914296468
Epoch:5  Train_loss:23.53226746045626   Test_loss:21.4284766515096
Epoch:6  Train_loss:20.295116238104992   Test_loss:20.51326986948649
Epoch:7  Train_loss:20.860542988165832   Test_loss:21.47391637166341
Epoch:8  Train_loss:19.656466050025745   Test_loss:20.58633778889974
Epoch:9  Train_loss:17.571220954259235   Test_loss:20.172656313578287
Epoch:10  Train_loss:17.936197326733517   Test_loss:20.17713673909505
Epoch:11  Train_loss:17.523160573763725   Test_loss:16.15451456705729
Epoch:12  Train_loss:16.305156937012306   Test_loss:15.621543884277344
Epoch:13  Train_loss:16.22855502519852   Test_loss:15.747962697347004
Epoch:14  Train_loss:16.190463439012184   Test_loss:16.494360605875652
Epoch:15  Train_loss:14.42